# POSE — `infer.ipynb` : ein Bild -> 6D-Pose

Nimmt **ein Bild aus `project/input/`** und lässt es durch die Inferenz-
Pipeline laufen, **jede Stufe sichtbar**:

| Stufe | Was | Output |
|---|---|---|
| 0 · Setup | Pfade + Modelle aus `project/models/` laden | — |
| 1 · Teile finden | **OBB-Detektor** (`models/detector.pt`, YOLOv8-OBB) -> orientierte Boxen + Klasse | Detections |
| 2 · Crop | pro Detection den BBox-Ausschnitt | Crops |
| 3 · 6D-Pose | **GDRNPP** (BOP-SOTA, ADR-018) -> `R_world`, `t_world` — angebunden in W2/W3 | Pose |
| 4 · pose_result | Schema-valides `pose_result.json` (Contract) + 3D-Inline + Live-Viewer | JSON + 3D |

**Konvention:** Z-up Welt, `world = R @ body` (Spaltenkonvention), Ursprung =
Tisch-Nullpunkt — eingefroren im Contract, nie transponiert.

> **6D-Pose:** Der alte Eigenbau-Mittelteil (Face-Classifier-CNN +
> Template-Bank-Render-and-Compare + Eigenbau-Backprojection) ist **bewusst
> entfernt** (ADR-018). Stufe 3 wird durch **GDRNPP** ersetzt (W2/W3). Detektor
> (Stufe 1) und der pose_result-Contract (Stufe 4) bleiben.

> **Detektor:** `models/detector.pt` ist ein echt trainierter YOLOv8-OBB-Detektor
> (YOLOv8s-OBB, 5 Klassen, mAP50 ≈ 0.99). Fehlt die `.pt` oder `ultralytics`,
> fällt Stufe 1 sauber auf die SDG-Annotator-Boxen (`bbox_2d_<idx>.json`) bzw.
> eine Dummy-Box zurück.

## 0 · Setup — Pfade + Modelle laden

In [ ]:
import os, sys, json, pathlib, glob
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from scipy import ndimage

PROJECT = pathlib.Path.cwd()
if PROJECT.name != "project":
    cand = PROJECT / "project"
    PROJECT = cand if cand.is_dir() else PROJECT
INPUT  = PROJECT / "input"
MODELS = PROJECT / "models"
TEMP   = PROJECT / "temp"
TEMP.mkdir(parents=True, exist_ok=True)

print("Modelle:", sorted(p.name for p in MODELS.glob("*.pt")) or "(keine .pt — Fallback wird genutzt)")
print("Input-Bilder:", sorted(p.name for p in INPUT.glob("*")) or "(leer — Stufe 1 legt ein Beispiel an)")

## 1 · Teile finden — OBB-Detektor (echt trainiert)

Der **YOLOv8-OBB-Detektor** (`models/detector.pt`) lokalisiert jedes Teil als
**orientierte Bounding-Box** + Klasse. Trainiert auf den SDG-Multi-Part-Szenen
(Stufe 5, 420 Multi-Part-Szenen, 5 Klassen) — YOLOv8s-OBB, mAP50≈0.99.

Drei Quellen, in dieser Priorität (jede ein sauberer Fallback der vorigen):

1. **OBB-Detektor** — `models/detector.pt` via `ultralytics`. Liefert orientierte
   4-Eck-Boxen, Klasse, Konfidenz und den **OBB-Winkel** (Lang-Achse), der die
   Yaw-Schätzung in Stufe 4 seedet.
2. **SDG-Annotator** — fehlt die `.pt`/`ultralytics`, nehmen wir
   `bbox_2d_<idx>.json` (Isaac `bounding_box_2d_tight`) neben dem Bild.
3. **Dummy** — sonst eine ganze-Bild-Box, damit die Kette nie hart bricht.

Das Beispielbild legen wir aus `input/` ab (bzw. bereitstellen, falls leer).


In [ ]:
# Ein Beispielbild bereitstellen, falls input/ leer ist.
def ensure_example_image():
    imgs = sorted(p for p in INPUT.glob("*") if p.suffix.lower() in (".png", ".jpg", ".jpeg"))
    if imgs: return imgs[0]
    # input/ ist leer: eine SDG-Szene reinlegen (setup.ipynb generate_scenes ->
    # data/output/big auf der Box, dann ein rgb_*.png nach project/input/ holen).
    print("[input] kein Bild in project/input/ — bitte eine SDG-Szene (rgb_*.png) "
          "ablegen (siehe setup.ipynb Stufe 1: generate_scenes).")
    return None

IMG_PATH = ensure_example_image()
print("Eingabebild:", IMG_PATH)


In [ ]:
# ── Detection-Quelle: OBB-Detektor (bevorzugt), sonst SDG-BBox-JSON, sonst Dummy
# Teile-Namen kommen aus models/part_meta.json (post-ADR-018), sonst Fallback-Liste.
_FALLBACK_PARTS = ["Anker_Lang","Anker_Kurz","Zahnrad","Poltopf_kurz_centered",
                   "Getriebegehaeuse_typ4","Buerstenhalter_2polig"]
def available_parts():
    pm = MODELS / "part_meta.json"
    if pm.exists():
        try:
            data = json.load(open(pm))
            names = list(data.get("parts", data)) if isinstance(data, (dict, list)) else []
            if names: return sorted(str(n) for n in names)
        except Exception:
            pass
    return list(_FALLBACK_PARTS)
def _canonical_parts():
    return {p.lower(): p for p in available_parts()}
_CANON = _canonical_parts()
def canonical_part(raw): return _CANON.get(raw.strip().lower(), raw)

# --- echter OBB-Detektor (models/detector.pt, lazy) ---------------------------
DETECTOR_FILE = MODELS / "detector.pt"
_DETECTOR = {}
def _load_detector():
    """YOLOv8-OBB lazy laden. None wenn keine .pt / kein ultralytics."""
    if "m" in _DETECTOR: return _DETECTOR["m"]
    m = None
    if DETECTOR_FILE.exists():
        try:
            from ultralytics import YOLO; m = YOLO(str(DETECTOR_FILE))
        except Exception as e:
            print(f"[detect] ultralytics/Checkpoint nicht ladbar ({e!r}) — Fallback")
    _DETECTOR["m"] = m; return m

def _obb_to_aabb(corners, W, H):
    xs=[c[0] for c in corners]; ys=[c[1] for c in corners]
    x0=max(0,min(int(round(min(xs))),W-1)); x1=max(0,min(int(round(max(xs))),W))
    y0=max(0,min(int(round(min(ys))),H-1)); y1=max(0,min(int(round(max(ys))),H))
    return [x0,y0,x1,y1]

def _obb_angle_deg(corners):
    p=np.asarray(corners,float); e01=p[1]-p[0]; e12=p[2]-p[1]
    le = e01 if np.hypot(*e01)>=np.hypot(*e12) else e12
    return float(np.degrees(np.arctan2(le[1], le[0])))

def detect_with_model(img_path):
    """Orientierte Detektionen via detector.pt. [] wenn nicht verfügbar."""
    m = _load_detector()
    if m is None: return []
    rgb = np.asarray(Image.open(img_path).convert("RGB")); H, W = rgb.shape[:2]
    try:
        r = m.predict(str(img_path), imgsz=1280, conf=0.40, verbose=False)[0]
    except Exception as e:
        print(f"[detect] Inferenz fehlgeschlagen ({e!r}) — Fallback"); return []
    if r.obb is None or len(r.obb)==0: return []
    polys=r.obb.xyxyxyxy.cpu().numpy(); cls=r.obb.cls.cpu().numpy().astype(int)
    conf=r.obb.conf.cpu().numpy(); names=r.names; dets=[]
    for inst,(poly,c,cf) in enumerate(zip(polys,cls,conf)):
        corners=[[float(x),float(y)] for x,y in poly]; bbox=_obb_to_aabb(corners,W,H)
        if (bbox[2]-bbox[0])<4 or (bbox[3]-bbox[1])<4: continue
        raw=names[int(c)]
        dets.append({"instance_id":inst,"part":canonical_part(raw),"bbox_2d":bbox,
                     "raw_label":raw,"occlusion":0.0,"obb_corners":corners,
                     "obb_angle_deg":_obb_angle_deg(corners),"det_conf":float(cf)})
    print(f"[detect] {len(dets)} orientierte Boxen vom OBB-Detektor (detector.pt)")
    return dets

# --- SDG-Annotator-Fallback ---------------------------------------------------
def find_bbox_json(img_path):
    img_path = pathlib.Path(img_path); stem = img_path.stem.replace("rgb_","").replace("scene_","")
    for cand in [img_path.parent / f"bbox_2d_{stem}.json", *sorted(img_path.parent.glob("bbox_2d_*.json"))]:
        if cand.exists(): return cand
    return None

def detections_for(img_path):
    """Detections fürs Bild: OBB-Detektor -> SDG-BBox-JSON -> Dummy."""
    rgb = np.asarray(Image.open(img_path).convert("RGB")); H, W = rgb.shape[:2]
    dets = detect_with_model(img_path)
    if dets: return rgb, dets
    bj = find_bbox_json(img_path)
    if bj is not None:
        data = json.load(open(bj)); rows = data.get("data", []); id2 = (data.get("info", {}) or {}).get("idToLabels", {})
        for inst, row in enumerate(rows):
            if len(row) < 5: continue
            raw = (id2.get(str(int(row[0])), {}) or {}).get("class", "")
            x0, x1 = sorted((int(round(row[1])), int(round(row[3]))))
            y0, y1 = sorted((int(round(row[2])), int(round(row[4]))))
            x0, x1 = max(0,min(x0,W-1)), max(0,min(x1,W)); y0, y1 = max(0,min(y0,H-1)), max(0,min(y1,H))
            if (x1-x0) < 4 or (y1-y0) < 4: continue
            dets.append({"instance_id": inst, "part": canonical_part(raw), "bbox_2d": [x0,y0,x1,y1],
                         "raw_label": raw, "occlusion": float(row[5]) if len(row) > 5 else 0.0})
        print(f"[detect] Fallback: {len(dets)} BBoxes aus SDG-Annotator {bj.name}")
    if not dets:
        part = available_parts()[0] if available_parts() else "Anker_Lang"
        dets = [{"instance_id": 0, "part": part, "bbox_2d": [0,0,W,H], "raw_label": part, "occlusion": 0.0}]
        print(f"[detect] DUMMY: eine ganze-Bild-BBox als '{part}'")
    return rgb, dets

RGB, DETS = detections_for(IMG_PATH)
print(f"Bild {RGB.shape}, {len(DETS)} Detections "
      f"({'OBB-Detektor' if DETS and 'obb_corners' in DETS[0] else 'SDG/Dummy'})")


### Visualisierung — BBox-Overlay

In [ ]:
def show_detections(rgb, dets):
    """OBB-Polygone (vom Detektor) bzw. achsenparallele Boxen (Fallback) zeichnen."""
    from matplotlib.patches import Rectangle, Polygon
    fig, ax = plt.subplots(figsize=(9, 6)); ax.imshow(rgb)
    for d in dets:
        if "obb_corners" in d:                              # orientierte Box vom Detektor
            ax.add_patch(Polygon(d["obb_corners"], closed=True, fill=False,
                                 edgecolor="deepskyblue", lw=1.6))
            cx = sum(c[0] for c in d["obb_corners"]) / 4; cy = sum(c[1] for c in d["obb_corners"]) / 4
            lbl = f"#{d['instance_id']} {d['part']} {d.get('det_conf',0):.2f}"
        else:                                               # achsenparallele Fallback-Box
            x0, y0, x1, y1 = d["bbox_2d"]
            ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False, edgecolor="lime", lw=1.5))
            cx, cy = x0, max(0, y0-3); lbl = f"#{d['instance_id']} {d['part']}"
        ax.text(cx, cy, lbl, color="white", fontsize=7, va="bottom",
                bbox=dict(boxstyle="round,pad=0.1", fc="black", alpha=0.55))
    src = "OBB-Detektor" if dets and "obb_corners" in dets[0] else "SDG/Dummy"
    ax.set_title(f"Detections ({src}) — {len(dets)} Teile"); ax.axis("off")
    plt.show()
show_detections(RGB, DETS)


## 2 · Crop — pro Detection den Ausschnitt

Der Crop ist das achsenparallele BBox-Rechteck pro Detektion — der
Eingabe-Ausschnitt fürs Pose-Backend (GDRNPP, Stufe 3). Liegt eine
semantische Segmentierung neben dem Bild, kann der Hintergrund auf
Neutralgrau gesetzt werden, damit das Teil statt Nachbar-Clutter dominiert;
ohne Seg: roher Crop.

In [ ]:
NEUTRAL = 128
def crop_detection(rgb, det):
    x0, y0, x1, y1 = det["bbox_2d"]
    return rgb[y0:y1, x0:x1].copy()

CROPS = [crop_detection(RGB, d) for d in DETS]

def show_crops(crops, dets, k=8):
    n = min(k, len(crops))
    if n == 0: print("keine Crops"); return
    fig, axs = plt.subplots(1, n, figsize=(2.0*n, 2.4), squeeze=False)
    for j in range(n):
        axs[0][j].imshow(crops[j]); axs[0][j].axis("off")
        axs[0][j].set_title(f"#{dets[j]['instance_id']} {dets[j]['part']}", fontsize=8)
    fig.suptitle("Crops", fontweight="bold"); fig.tight_layout(); plt.show()
show_crops(CROPS, DETS)


## 3 · 6D-Pose — GDRNPP (BOP-SOTA)

# === BOP pose pipeline (GDRNPP — siehe ADR-018), wird in W2/W3 eingesetzt ===

Der alte Eigenbau-Mittelteil (Face-Classifier-CNN, `faces_<part>.json`-
Registry, Template-Bank-Render-and-Compare mit Yaw-Suche per Silhouetten-MSE,
Eigenbau-Backprojection) ist **bewusst entfernt** (ADR-018, BOP-Pivot — „für
die Tonne"). Rotation + Translation kommen künftig aus **GDRNPP**: pro
Detektion liefert es `R_world` + `t_world` direkt aus dem BOP-Modell. Bis die
GDRNPP-Inferenz in W2/W3 angebunden ist, ist die Stufe ein **Stub** (keine
Posen) — der pose_result-Contract bleibt unverändert und schema-valide.

In [ ]:
# === BOP pose pipeline (GDRNPP — siehe ADR-018), wird in W2/W3 eingesetzt ===
# STUB: GDRNPP ist noch nicht angebunden. Bis dahin keine Posen -> leeres
# Ergebnis. Sobald GDRNPP läuft, füllt es ALIGNED je Detektion mit:
#   {instance_id, part, face_name, R_world(9), t_world(3), upright, confidence, bbox_2d(4)}
# und die nächste Zelle giesst das schema-valide in pose_result.json.
class _TableOrigin:
    """Platzhalter bis GDRNPP die metrische Kamera/Backprojection liefert."""
    table_origin = (0.0, 0.0, 0.08)   # Tisch-Nullpunkt (Welt, Meter)
INTR = _TableOrigin()
ALIGNED = []   # GDRNPP-Output (W2/W3); leer = noch kein Pose-Backend
print(f"[pose] GDRNPP-Stub: {len(ALIGNED)} Posen "
      f"(BOP-Pipeline wird in W2/W3 angebunden — ADR-018)")

## 4 · pose_result — schema-valides JSON + 3D-Inline

Die Ergebnisse werden in das eingefrorene `pose_result`-Format gegossen und gegen
den Contract validiert (stdlib-Gate immer; jsonschema-Gate wenn verfügbar).


In [ ]:
SCHEMA_VERSION = "1.0.0"
COORD_CONVENTION = ("Z-up world; column rotation world = R @ body; "
                    "origin = table-plane null-point")

def _is_num(x): return isinstance(x, (int, float)) and not isinstance(x, bool)
def _is_int(x): return isinstance(x, int) and not isinstance(x, bool)

def check_pose_result(doc):
    """stdlib-Contract-Gate -> Liste von Verstößen (leer = gültig)."""
    e = []
    if not isinstance(doc, dict): return ["top level must be object"]
    meta = doc.get("meta", {})
    if not isinstance(meta.get("source_image"), str) or not meta.get("source_image"): e.append("meta.source_image")
    to = meta.get("table_origin")
    if not (isinstance(to, list) and len(to) == 3 and all(_is_num(v) for v in to)): e.append("meta.table_origin")
    if meta.get("units") != "m": e.append("meta.units must be 'm'")
    if not isinstance(meta.get("coordinate_convention"), str) or not meta.get("coordinate_convention"): e.append("meta.coordinate_convention")
    sv = meta.get("schema_version")
    if not (isinstance(sv, str) and sv.count(".") == 2 and all(p.isdigit() for p in sv.split("."))): e.append("meta.schema_version")
    res = doc.get("results")
    if not isinstance(res, list): return e + ["results must be array"]
    for i, r in enumerate(res):
        p = f"results[{i}]"
        if not (_is_int(r.get("instance_id")) and r["instance_id"] >= 0): e.append(f"{p}.instance_id")
        if not (isinstance(r.get("part"), str) and r["part"]): e.append(f"{p}.part")
        if not (isinstance(r.get("face"), str) and r["face"]): e.append(f"{p}.face")
        R = r.get("R_world")
        if not (isinstance(R, list) and len(R) == 9 and all(_is_num(v) for v in R)): e.append(f"{p}.R_world")
        t = r.get("t_world")
        if not (isinstance(t, list) and len(t) == 3 and all(_is_num(v) for v in t)): e.append(f"{p}.t_world")
        c = r.get("confidence")
        if not (_is_num(c) and 0.0 <= c <= 1.0): e.append(f"{p}.confidence")
        b = r.get("bbox_2d")
        if not (isinstance(b, list) and len(b) == 4 and all(_is_int(v) and v >= 0 for v in b)): e.append(f"{p}.bbox_2d")
        elif not (b[0] <= b[2] and b[1] <= b[3]): e.append(f"{p}.bbox_2d order")
        if not isinstance(r.get("upright"), bool): e.append(f"{p}.upright")
    return e

def build_pose_result(img_path, aligned, intr):
    return {
        "meta": {"source_image": str(img_path), "table_origin": [float(v) for v in intr.table_origin],
                 "units": "m", "coordinate_convention": COORD_CONVENTION, "schema_version": SCHEMA_VERSION},
        "results": [{"instance_id": int(a["instance_id"]), "part": a["part"], "face": a["face_name"],
                     "R_world": [float(v) for v in a["R_world"]], "t_world": [float(v) for v in a["t_world"]],
                     "confidence": float(max(0.0, min(1.0, a["confidence"]))),
                     "bbox_2d": [int(v) for v in a["bbox_2d"]], "upright": bool(a["upright"])}
                    for a in aligned],
    }

POSE = build_pose_result(IMG_PATH, ALIGNED, INTR)
errors = check_pose_result(POSE)
try:
    from jsonschema import Draft202012Validator
    # Contract liegt in project/ (committet); ältere docs/-Lage als Fallback.
    schema_file = next((p for p in (PROJECT / "pose_result.schema.json",
                                    PROJECT.parent / "docs" / "pose_result.schema.json")
                        if p.exists()), None)
    if schema_file:
        v = Draft202012Validator(json.load(open(schema_file)))
        errors += [f"[jsonschema] {'/'.join(str(x) for x in e.path)}: {e.message}" for e in v.iter_errors(POSE)]
except Exception:
    pass
assert not errors, f"pose_result NICHT contract-valid:\n  - " + "\n  - ".join(errors)
out = TEMP / "pose_result.json"; json.dump(POSE, open(out, "w"), indent=2)
print(f"[pose_result] {len(POSE['results'])} Teile, Schema-Gate PASS -> {out}")
print(json.dumps(POSE, indent=2)[:900])

### Visualisierung — 3D-Inline (Teile an ihrer geschätzten Pose)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa
def show_3d(pose):
    fig = plt.figure(figsize=(7, 6)); ax = fig.add_subplot(111, projection="3d")
    # Tischebene
    gx, gy = np.meshgrid(np.linspace(-0.1, 0.1, 2), np.linspace(-0.1, 0.1, 2))
    ax.plot_surface(gx, gy, np.zeros_like(gx), alpha=0.1, color="grey")
    for r in pose["results"]:
        t = np.array(r["t_world"]); R = np.array(r["R_world"]).reshape(3, 3)
        for axis, col in zip(range(3), ("r", "g", "b")):
            d = R[:, axis]*0.03
            ax.quiver(t[0], t[1], t[2], d[0], d[1], d[2], color=col, lw=2)
        ax.text(t[0], t[1], t[2]+0.01, f"{r['part']}\n{r['face']}", fontsize=7)
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.set_title("Geschätzte 6D-Posen (Body-Achsen je Teil)"); plt.show()
show_3d(POSE)


## 5 · Live-3D-Viewer — localhost-Server + IFrame im Notebook

Das `pose_result.json` (in `temp/`) ist der **Contract** zwischen Pipeline und
3D-Viewer (`project/frontend/`). Der Viewer lädt das **echte Anlagen-CAD**
(`frontend/assets/cell.glb` — Tisch/Wagen + NEURA-LARA5-Arm + Trays) und legt die
rekonstruierten Teile per 6D-Pose relativ zum Tisch-Nullpunkt drauf. Die letzte
Zelle startet einen localhost-Server (ab `project/`, im Hintergrund) und bettet
den Viewer als IFrame direkt ins Notebook ein.

> **Stand (ADR-018):** Detektor läuft echt (YOLOv8-OBB); die
> 6D-Pose kommt aus **GDRNPP** (BOP-SOTA), angebunden in W2/W3. Der
> Viewer + der pose_result-Contract bleiben unverändert.

In [ ]:
# ── localhost-Server (Hintergrund, ab project/) + Viewer-IFrame im Notebook ──
import http.server, socketserver, threading, functools, contextlib
from IPython.display import IFrame, display

PORT = 8000
SERVE_ROOT = PROJECT                       # Server-Wurzel = project/  -> /frontend/ + /temp/
POSE_REL = (TEMP / "pose_result.json").relative_to(PROJECT).as_posix()   # temp/pose_result.json
VIEWER_URL = f"http://127.0.0.1:{PORT}/frontend/?file=../{POSE_REL}"

def _start_server(root, port):
    """Idempotenter Hintergrund-Server. No-op, wenn der Port schon belegt ist."""
    handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=str(root))
    try:
        socketserver.TCPServer.allow_reuse_address = True
        httpd = socketserver.TCPServer(("127.0.0.1", port), handler)
    except OSError:
        print(f"[serve] Port {port} schon belegt — nehme den laufenden Server an.")
        return None
    threading.Thread(target=httpd.serve_forever, daemon=True).start()
    print(f"[serve] http.server läuft (Hintergrund) ab {root} auf :{port}")
    return httpd

_HTTPD = _start_server(SERVE_ROOT, PORT)
print(f"[serve] Viewer-URL: {VIEWER_URL}")
print(f"[serve] {len(POSE['results'])} Teile im pose_result -> wird unten gerendert.")
# Three.js-Viewer direkt ins Notebook einbetten (zeigt Tisch + Teile an 6D-Pose):
display(IFrame(src=VIEWER_URL, width="100%", height=560))
